# 🤖 TelecomX — Modelado Predictivo de Evasión de Clientes

---

**Parte 2 del desafío:** Preprocesamiento avanzado, análisis de correlación y modelos de Machine Learning para predecir el churn.

| | |
|---|---|
| **Dataset** | `TelecomX_limpio.csv` (salida de la Parte 1) |
| **Objetivo** | Predecir si un cliente cancelará el servicio |
| **Modelos** | Regresión Logística · Random Forest · KNN · Árbol de Decisión |
| **Métricas** | Accuracy · Precisión · Recall · F1 · Matriz de Confusión |


---
## ⚙️ 0. Importaciones y Configuración


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

# Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# Evaluación
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.05)
AZUL, ROJO = '#4C9BE8', '#E8634C'

print('✅ Librerías cargadas correctamente')


---
## 📂 1. Carga del Archivo Tratado

Cargamos el CSV generado en la Parte 1, que ya contiene los datos limpios, estandarizados y con las columnas renombradas al español.


In [ ]:
df = pd.read_csv('TelecomX_limpio.csv')

print(f'Shape: {df.shape}')
print(f'Columnas: {df.columns.tolist()}')
display(df.head(3))


---
## 🗑️ 2. Eliminación de Columnas Irrelevantes

Eliminamos `ID_Cliente` ya que es un identificador único que no aporta información predictiva. Incluirlo podría causar que el modelo memorice patrones inexistentes asociados a IDs específicos.


In [ ]:
df = df.drop(columns=['ID_Cliente'])

print(f'✅ Columna ID_Cliente eliminada.')
print(f'   Shape resultante: {df.shape}')
print(f'   Columnas restantes: {df.columns.tolist()}')


---
## 🔢 3. Encoding de Variables Categóricas

Los algoritmos de Machine Learning requieren datos numéricos. Aplicamos **One-Hot Encoding** con `pd.get_dummies()` a las columnas categóricas restantes.

### ¿Por qué get_dummies y no OneHotEncoder?
- `pd.get_dummies()` es más simple e integrado con DataFrames de pandas.
- `OneHotEncoder` de scikit-learn es preferible en pipelines de producción (evita data leakage).
- Para este análisis exploratorio, `get_dummies` con `drop_first=True` es suficiente y evita la **multicolinealidad** (dummy variable trap).


In [ ]:
# Identificar columnas categóricas que aún necesitan encoding
cols_cat = df.select_dtypes(include=['object', 'str']).columns.tolist()
print(f'Columnas categóricas a codificar: {cols_cat}')

# One-Hot Encoding — drop_first=True evita multicolinealidad
df_encoded = pd.get_dummies(df, columns=cols_cat, drop_first=True, dtype=int)

print(f'\n✅ Encoding completado.')
print(f'   Shape antes:  {df.shape}')
print(f'   Shape después: {df_encoded.shape}')
print(f'\n   Columnas generadas por encoding:')
nuevas = [c for c in df_encoded.columns if c not in df.columns]
for c in nuevas:
    print(f'   • {c}')


In [ ]:
display(df_encoded.head(3))


---
## ⚖️ 4. Verificación de la Proporción de Cancelación (Churn)

Antes de entrenar cualquier modelo, debemos entender el balance entre las clases. Un dataset desbalanceado puede hacer que el modelo aprenda a predecir siempre la clase mayoritaria.


In [ ]:
conteo = df_encoded['Evasion'].value_counts()
prop   = df_encoded['Evasion'].value_counts(normalize=True).mul(100).round(2)

print('📊 Distribución de la variable objetivo (Evasion):')
print(f'   0 (No evadió): {conteo[0]:>5} clientes  ({prop[0]:.1f}%)')
print(f'   1 (Evadió):    {conteo[1]:>5} clientes  ({prop[1]:.1f}%)')
print(f'\n⚠️  Ratio de desbalance: {conteo[0]/conteo[1]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Proporción de Evasión', fontsize=13, fontweight='bold')

colores = [AZUL, ROJO]
bars = axes[0].bar(['No Evadió (0)', 'Evadió (1)'], conteo.values,
                    color=colores, edgecolor='white', width=0.5)
for bar, v, p in zip(bars, conteo.values, prop.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{v:,}\n({p}%)', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Cantidad de clientes')
axes[0].set_ylim(0, conteo.max() * 1.2)

axes[1].pie(conteo.values, labels=['No Evadió', 'Evadió'], colors=colores,
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Proporción')

plt.tight_layout()
plt.show()

print('\n💡 Conclusión: existe desbalance de clases (73.5% vs 26.5%).')
print('   Se aplicará oversampling de la clase minoritaria (resample) en el entrenamiento.')


---
## 📏 5. Normalización / Estandarización

### ¿Cuándo es necesaria?
| Modelo | Sensible a escala | Requiere normalización |
|---|---|---|
| Regresión Logística | ✅ Sí | ✅ Sí |
| KNN | ✅ Sí | ✅ Sí |
| Árbol de Decisión | ❌ No | ❌ No |
| Random Forest | ❌ No | ❌ No |

Crearemos **dos versiones** del dataset: con y sin estandarización, y usaremos la versión correcta para cada modelo.

`StandardScaler` transforma cada variable para que tenga **media=0 y desviación estándar=1**, evitando que variables con magnitudes grandes dominen el aprendizaje.


In [ ]:
# Separar features y variable objetivo ANTES de escalar
X = df_encoded.drop(columns=['Evasion'])
y = df_encoded['Evasion']

print(f'Features (X): {X.shape}')
print(f'Target  (y): {y.shape}')
print(f'\nVariables numéricas que se escalarán:')
num_cols = ['Meses_Contrato', 'Cargo_Mensual', 'Cargo_Total', 'Cuentas_Diarias']
display(X[num_cols].describe().round(2))


---
## 🔗 6. Análisis de Correlación

La matriz de correlación permite identificar qué variables tienen mayor relación lineal con la evasión (`Evasion`). Correlaciones altas (positivas o negativas) indican variables con mayor poder predictivo. También detectamos **multicolinealidad** entre features.


In [ ]:
# Correlación con la variable objetivo — ordenada de mayor a menor
corr_target = df_encoded.corr(numeric_only=True)['Evasion'].drop('Evasion').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# — Barras: correlación con Evasion —
colors_bar = [ROJO if v > 0 else AZUL for v in corr_target.values]
axes[0].barh(corr_target.index, corr_target.values, color=colors_bar,
             edgecolor='white', linewidth=0.5)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Correlación de cada variable con Evasión', fontweight='bold')
axes[0].set_xlabel('Coeficiente de Pearson')

# — Heatmap de correlación completa —
corr_matrix = df_encoded.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=axes[1], cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.3,
            annot=False, cbar_kws={'shrink': 0.8})
axes[1].set_title('Matriz de Correlación (triángulo inferior)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)
axes[1].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.show()

print('\n📊 Top 10 variables con mayor correlación (absoluta) con Evasión:')
display(corr_target.abs().sort_values(ascending=False).head(10).to_frame('|Correlación|').round(4))


---
## 🔍 7. Análisis Dirigido

Exploramos en detalle cómo las variables más relevantes se relacionan con la evasión, usando boxplots y scatter plots.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Análisis Dirigido: Variables Clave vs Evasión', fontsize=14, fontweight='bold')

evasion_label = df_encoded['Evasion'].map({0: 'No', 1: 'Sí'})

# — Boxplot: Meses de Contrato vs Evasión —
for label, color in [('No', AZUL), ('Sí', ROJO)]:
    subset = df_encoded[evasion_label == label]['Meses_Contrato']
    axes[0,0].boxplot(subset, positions=[0 if label=='No' else 1],
                      patch_artist=True,
                      boxprops=dict(facecolor=color, alpha=0.7),
                      medianprops=dict(color='black', linewidth=2),
                      whiskerprops=dict(linewidth=1.2),
                      capprops=dict(linewidth=1.2),
                      flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[0,0].set_xticks([0, 1])
axes[0,0].set_xticklabels(['No Evadió', 'Evadió'])
axes[0,0].set_title('Meses de Contrato × Evasión', fontweight='bold')
axes[0,0].set_ylabel('Meses de Contrato')

# — Boxplot: Cargo Total vs Evasión —
for label, color in [('No', AZUL), ('Sí', ROJO)]:
    subset = df_encoded[evasion_label == label]['Cargo_Total']
    axes[0,1].boxplot(subset, positions=[0 if label=='No' else 1],
                      patch_artist=True,
                      boxprops=dict(facecolor=color, alpha=0.7),
                      medianprops=dict(color='black', linewidth=2),
                      whiskerprops=dict(linewidth=1.2),
                      capprops=dict(linewidth=1.2),
                      flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[0,1].set_xticks([0, 1])
axes[0,1].set_xticklabels(['No Evadió', 'Evadió'])
axes[0,1].set_title('Cargo Total × Evasión', fontweight='bold')
axes[0,1].set_ylabel('Cargo Total ($)')

# — Scatter: Meses de Contrato vs Cargo Total —
for label, color in [('No', AZUL), ('Sí', ROJO)]:
    mask = evasion_label == label
    axes[1,0].scatter(df_encoded[mask]['Meses_Contrato'],
                      df_encoded[mask]['Cargo_Total'],
                      c=color, alpha=0.2, s=8, label=label)
axes[1,0].set_title('Meses de Contrato vs Cargo Total', fontweight='bold')
axes[1,0].set_xlabel('Meses de Contrato')
axes[1,0].set_ylabel('Cargo Total ($)')
axes[1,0].legend(title='Evasión')

# — KDE: Cargo Mensual vs Evasión —
for label, color in [('No', AZUL), ('Sí', ROJO)]:
    subset = df_encoded[evasion_label == label]['Cargo_Mensual']
    axes[1,1].hist(subset, bins=30, color=color, alpha=0.5, label=label, density=True)
    axes[1,1].axvline(subset.mean(), color=color, linestyle='--', linewidth=1.5)
axes[1,1].set_title('Distribución de Cargo Mensual', fontweight='bold')
axes[1,1].set_xlabel('Cargo Mensual ($)')
axes[1,1].legend(title='Evasión')

plt.tight_layout()
plt.show()


In [ ]:
print('📊 Estadísticas por grupo:')
display(df_encoded.groupby('Evasion')[['Meses_Contrato','Cargo_Mensual','Cargo_Total']]
        .agg(['mean','median']).round(2))


---
## ✂️ 8. Separación de Datos y Balanceo de Clases

### División 80/20
Usamos 80% para entrenamiento y 20% para prueba. `stratify=y` garantiza que la proporción de evasión sea igual en ambos conjuntos.

### Balanceo con Oversampling
Para contrarrestar el desbalance (73.5% vs 26.5%), aplicamos **oversampling** de la clase minoritaria en el conjunto de entrenamiento. **Importante:** el balanceo se aplica SOLO al set de entrenamiento, nunca al de prueba.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ División completada:')
print(f'   Entrenamiento: {X_train.shape[0]:,} muestras')
print(f'   Prueba:        {X_test.shape[0]:,} muestras')
print(f'\n   Proporción en train — No evadió: {(y_train==0).sum()}, Evadió: {(y_train==1).sum()}')
print(f'   Proporción en test  — No evadió: {(y_test==0).sum()}, Evadió: {(y_test==1).sum()}')

# Oversampling de la clase minoritaria en entrenamiento
X_train_df = X_train.copy()
X_train_df['Evasion'] = y_train.values

mayoria  = X_train_df[X_train_df['Evasion'] == 0]
minoria  = X_train_df[X_train_df['Evasion'] == 1]

minoria_ups = resample(minoria, replace=True, n_samples=len(mayoria), random_state=42)
train_bal   = pd.concat([mayoria, minoria_ups]).sample(frac=1, random_state=42)

X_train_bal = train_bal.drop(columns=['Evasion'])
y_train_bal = train_bal['Evasion']

print(f'\n✅ Balanceo completado (oversampling):')
print(f'   No evadió: {(y_train_bal==0).sum():,}  |  Evadió: {(y_train_bal==1).sum():,}')


In [ ]:
# Estandarización para modelos sensibles a escala (LR, KNN)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal)   # fit SOLO en train
X_test_sc  = scaler.transform(X_test)

print('✅ StandardScaler aplicado:')
print('   • X_train_bal → X_train_sc  (para LR y KNN)')
print('   • X_test      → X_test_sc   (para evaluación de LR y KNN)')
print('   • X_train_bal y X_test se usan SIN escalar para DT y RF')


---
## 🤖 9. Creación y Entrenamiento de Modelos

Entrenamos cuatro modelos que cubren distintos enfoques algorítmicos:

| Modelo | Tipo | Normalización | Interpretabilidad |
|---|---|---|---|
| Regresión Logística | Lineal | ✅ Sí | Alta (coeficientes) |
| KNN | Basado en distancia | ✅ Sí | Baja |
| Árbol de Decisión | Basado en árboles | ❌ No | Alta (visualizable) |
| Random Forest | Ensemble de árboles | ❌ No | Media (importancia) |


In [ ]:
# ── Modelo 1: Regresión Logística (con normalización) ──
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train_bal)
y_pred_lr = lr.predict(X_test_sc)
print('✅ Regresión Logística entrenada')

# ── Modelo 2: KNN (con normalización) ──
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_sc, y_train_bal)
y_pred_knn = knn.predict(X_test_sc)
print('✅ KNN (k=7) entrenado')

# ── Modelo 3: Árbol de Decisión (sin normalización) ──
dt = DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=42)
dt.fit(X_train_bal, y_train_bal)
y_pred_dt = dt.predict(X_test)
print('✅ Árbol de Decisión entrenado')

# ── Modelo 4: Random Forest (sin normalización) ──
rf = RandomForestClassifier(n_estimators=100, max_depth=8,
                             min_samples_leaf=10, random_state=42, n_jobs=-1)
rf.fit(X_train_bal, y_train_bal)
y_pred_rf = rf.predict(X_test)
print('✅ Random Forest entrenado')


---
## 📈 10. Evaluación de Modelos

### Métricas utilizadas
- **Accuracy**: % de predicciones correctas en total.
- **Precisión**: de todos los que predijo como evasión, ¿cuántos realmente evadieron?
- **Recall**: de todos los que realmente evadieron, ¿cuántos detectó el modelo?
- **F1-Score**: media armónica entre Precisión y Recall. Ideal para clases desbalanceadas.

> 💡 En problemas de churn, el **Recall** es especialmente importante: preferimos detectar a todos los que van a evadir, aunque generemos algunas falsas alarmas.


In [ ]:
modelos = {
    'Regresión Logística': y_pred_lr,
    'KNN (k=7)':           y_pred_knn,
    'Árbol de Decisión':   y_pred_dt,
    'Random Forest':       y_pred_rf,
}

resultados = []
for nombre, y_pred in modelos.items():
    resultados.append({
        'Modelo':     nombre,
        'Accuracy':   round(accuracy_score(y_test, y_pred), 4),
        'Precisión':  round(precision_score(y_test, y_pred), 4),
        'Recall':     round(recall_score(y_test, y_pred), 4),
        'F1-Score':   round(f1_score(y_test, y_pred), 4),
    })

df_res = pd.DataFrame(resultados).set_index('Modelo')
print('📊 Tabla comparativa de métricas:')
display(df_res.style
    .highlight_max(color='#d4edda', axis=0)
    .highlight_min(color='#f8d7da', axis=0)
    .format('{:.4f}')
)


In [ ]:
# ── Matrices de confusión ──
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
fig.suptitle('Matrices de Confusión', fontsize=14, fontweight='bold')

for ax, (nombre, y_pred) in zip(axes, modelos.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Evadió', 'Evadió'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nombre, fontweight='bold', fontsize=9)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.show()


In [ ]:
# ── Comparación visual de métricas ──
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(df_res.columns))
width = 0.2
colores_mod = ['#2E86C1', '#E74C3C', '#27AE60', '#8E44AD']

for i, (nombre, fila) in enumerate(df_res.iterrows()):
    bars = ax.bar(x + i * width, fila.values, width,
                  label=nombre, color=colores_mod[i], alpha=0.85, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_res.columns, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Valor de la métrica')
ax.set_title('Comparación de Modelos — Todas las Métricas', fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
plt.show()


In [ ]:
# ── Reporte detallado del mejor modelo ──
mejor = df_res['F1-Score'].idxmax()
print(f'🏆 Mejor modelo por F1-Score: {mejor}')
print()
y_mejor = modelos[mejor]
print(classification_report(y_test, y_mejor, target_names=['No Evadió', 'Evadió']))


### Análisis de Overfitting / Underfitting

Comparamos el F1-Score en entrenamiento vs prueba para detectar si algún modelo sobreajusta.


In [ ]:
print('📊 F1-Score en Train vs Test (detección de overfitting):')
print(f'{'Modelo':<25} {'F1 Train':>10} {'F1 Test':>10} {'Diferencia':>12}')
print('-' * 60)

configs = [
    ('Regresión Logística', lr,  X_train_sc, y_train_bal, X_test_sc,  y_test),
    ('KNN (k=7)',           knn, X_train_sc, y_train_bal, X_test_sc,  y_test),
    ('Árbol de Decisión',  dt,  X_train_bal, y_train_bal, X_test,    y_test),
    ('Random Forest',      rf,  X_train_bal, y_train_bal, X_test,    y_test),
]

for nombre, modelo, Xtr, ytr, Xte, yte in configs:
    f1_tr = f1_score(ytr, modelo.predict(Xtr))
    f1_te = f1_score(yte, modelo.predict(Xte))
    diff  = f1_tr - f1_te
    alerta = ' ⚠️ posible overfit' if diff > 0.1 else ''
    print(f'{nombre:<25} {f1_tr:>10.4f} {f1_te:>10.4f} {diff:>+12.4f}{alerta}')


---
## 🎯 11. Análisis de Importancia de Variables

Cada modelo expone la relevancia de las variables de manera diferente:
- **Regresión Logística**: coeficientes (magnitud = importancia, signo = dirección)
- **KNN**: no tiene importancia directa; analizamos las variables más correlacionadas con las predicciones
- **Árbol de Decisión**: feature importance por reducción de impureza (Gini)
- **Random Forest**: feature importance promediada sobre todos los árboles


In [ ]:
feature_names = X.columns.tolist()
TOP_N = 15

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle(f'Top {TOP_N} Variables más Relevantes por Modelo', fontsize=14, fontweight='bold')

# ── Regresión Logística: coeficientes ──
coef_lr = pd.Series(np.abs(lr.coef_[0]), index=feature_names).sort_values(ascending=False).head(TOP_N)
axes[0].barh(coef_lr.index[::-1], coef_lr.values[::-1], color='#2E86C1', edgecolor='white')
axes[0].set_title('Regresión Logística\n(|coeficientes|)', fontweight='bold')
axes[0].set_xlabel('|Coeficiente|')

# ── Árbol de Decisión: feature importance ──
imp_dt = pd.Series(dt.feature_importances_, index=feature_names).sort_values(ascending=False).head(TOP_N)
axes[1].barh(imp_dt.index[::-1], imp_dt.values[::-1], color='#27AE60', edgecolor='white')
axes[1].set_title('Árbol de Decisión\n(Importancia Gini)', fontweight='bold')
axes[1].set_xlabel('Importancia')

# ── Random Forest: feature importance ──
imp_rf = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False).head(TOP_N)
axes[2].barh(imp_rf.index[::-1], imp_rf.values[::-1], color='#8E44AD', edgecolor='white')
axes[2].set_title('Random Forest\n(Importancia promediada)', fontweight='bold')
axes[2].set_xlabel('Importancia')

plt.tight_layout()
plt.show()


In [ ]:
# Coeficientes LR con signo (dirección del efecto)
coef_signed = pd.Series(lr.coef_[0], index=feature_names).sort_values()
fig, ax = plt.subplots(figsize=(10, 8))
colors_coef = [ROJO if v > 0 else AZUL for v in coef_signed.values]
ax.barh(coef_signed.index, coef_signed.values, color=colors_coef, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Regresión Logística — Coeficientes con dirección\n'             '(Rojo = aumenta riesgo de evasión | Azul = reduce riesgo)', fontweight='bold')
ax.set_xlabel('Coeficiente')
plt.tight_layout()
plt.show()


In [ ]:
# Variables top coincidentes entre los tres modelos
top_lr = set(coef_lr.head(10).index)
top_dt = set(imp_dt.head(10).index)
top_rf = set(imp_rf.head(10).index)

print('🔑 Variables en el Top-10 de los TRES modelos simultáneamente:')
consenso = top_lr & top_dt & top_rf
for v in sorted(consenso):
    print(f'   ✅ {v}')

print('\n🔑 Variables en el Top-10 de DOS modelos:')
dos = (top_lr & top_dt) | (top_lr & top_rf) | (top_dt & top_rf) - consenso
for v in sorted(dos):
    print(f'   🟡 {v}')


---
## 📝 12. Conclusión Final

### Comparación de Modelos

| Modelo | Fortaleza | Debilidad |
|---|---|---|
| **Regresión Logística** | Interpretable, rápida | Asume linealidad |
| **KNN** | No paramétrico | Lento en producción, sensible a ruido |
| **Árbol de Decisión** | Muy interpretable | Propenso a overfitting |
| **Random Forest** | Alta precisión, robusto | Menos interpretable |

### Factores que más influyen en la evasión

Basándonos en la convergencia de los tres modelos interpretables, los factores con mayor impacto en la predicción de churn son:

1. **Tipo de contrato Mes a Mes** — el predictor más fuerte de evasión.
2. **Meses de contrato (tenure)** — clientes con menos meses evaden más.
3. **Cargo mensual alto** — mayores cargos correlacionan con mayor evasión.
4. **Método de pago: Cheque Electrónico** — señal clara de desenganche.
5. **Fibra Óptica como tipo de internet** — paradoja precio-valor.

### Estrategias de Retención recomendadas

- 🎯 **Migrar contratos** Mes a Mes a modalidades anuales con incentivos.
- 🚀 **Intervenir tempranamente** en los primeros 6 meses de contrato.
- 💳 **Incentivar el pago automático** con descuentos por débito/tarjeta.
- 🔧 **Mejorar la calidad percibida** del servicio de Fibra Óptica.
- 🤖 **Implementar el modelo predictivo** (Random Forest) para identificar clientes en riesgo antes de que cancelen y activar acciones preventivas.

---
*Análisis completado — TelecomX Churn Prediction · 2025*
